# SmoothQuant full-quant notebook — Perplexity + ARC-Challenge

This notebook follows the same staged flow as your GPTQ notebook:

1. build the compressed artifact
2. evaluate perplexity
3. evaluate ARC-Challenge
4. save metrics, predictions, manifests, and zip bundles

Implementation note:
This is a **self-contained SmoothQuant notebook**. Instead of depending on fragile research repos or custom CUDA kernels, it implements a Colab-safe W8A8 simulation directly in PyTorch. The wrapper applies the SmoothQuant scaling transform, uses per-row static int8 weights, and dynamic per-token int8 activations. That makes the notebook easier to run and reload, while still matching the activation-quantization goal of SmoothQuant.


In [ ]:

# Colab-safe environment setup
# This notebook avoids fragile research repos and uses only Hugging Face / PyTorch primitives.
# Restart the runtime only if pip upgrades core packages in your environment.

!pip -q install -U pip setuptools wheel
!pip -q install "transformers>=4.44,<4.58" "datasets>=3.0,<4.0" "accelerate>=0.30,<1.0" pandas pyarrow safetensors tqdm sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:

PROJECT_ROOT = Path.cwd().resolve()
VARIANT = 'full_quant'
RUN_NAME = 'smoothquant_full_quant_ppl_arc'
TECHNIQUE_NAME = 'SmoothQuant'
TECHNIQUE_DIR = 'smoothquant'
BASE_MODEL = 'Qwen/Qwen2-1.5B'

cfg = {
    'base_model': BASE_MODEL,
    'calibration': {
        'dataset': 'wikitext',
        'dataset_name': 'wikitext-2-raw-v1',
        'split': 'train',
        'num_samples': 512,
        'max_length': 2048,
        'seed': 42,
    },
    'smoothquant': {
        'alpha': 0.80,
    },
    'eval': {
        'generation_max_input_length': 1792,
        'perplexity': {
            'dataset': 'wikitext',
            'dataset_name': 'wikitext-2-raw-v1',
            'split': 'test',
            'max_length': 2048,
            'stride': 1024,
            'max_eval_tokens': 131072,
            'log_every': 25,
        },
        'arc_challenge': {
            'num_samples': 500,
            'max_new_tokens': 5,
            'log_every': 50,
        },
    },
    'paths': {
        'artifacts_dir': str(PROJECT_ROOT / 'artifacts'),
        'results_dir': str(PROJECT_ROOT / 'results' / RUN_NAME),
        'calibration_dir': str(PROJECT_ROOT / 'calibration_data'),
        'bundles_dir': str(PROJECT_ROOT / 'zip_bundles' / RUN_NAME),
    },
}
print(json.dumps(cfg, indent=2))


In [ ]:

from __future__ import annotations

import gc
import json
import logging
import math
import os
import random
import re
import shutil
import sys
import time
import zipfile
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, set_seed

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger(RUN_NAME)

PROJECT_ROOT = Path.cwd().resolve()
set_seed(42)
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

def ensure_pad_token(tokenizer):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

PATHS = {k: Path(v) for k, v in cfg['paths'].items()}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIR = PATHS['artifacts_dir'] / TECHNIQUE_DIR / VARIANT
RESULTS_DIR = PATHS['results_dir'] / VARIANT
BUNDLE_DIR = PATHS['bundles_dir'] / VARIANT
CALIB_DIR = PATHS['calibration_dir']
for p in [ARTIFACT_DIR, RESULTS_DIR, BUNDLE_DIR, CALIB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def save_json(obj: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)


def read_json(path: Path, default=None):
    if not path.exists():
        return {} if default is None else default
    with open(path, 'r') as f:
        return json.load(f)


def zip_directory(src_dir: Path, zip_path: Path) -> Path:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for file in src_dir.rglob('*'):
            if file.is_file() and file != zip_path:
                zf.write(file, file.relative_to(src_dir.parent))
    return zip_path


def artifact_size_gb(path: Path) -> float:
    total = 0
    for p in path.rglob('*'):
        if p.is_file():
            total += p.stat().st_size
    return total / (1024 ** 3)


def format_seconds(seconds: float) -> str:
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def get_model_device(model: nn.Module) -> torch.device:
    return next(model.parameters()).device


def find_parent_module(model: nn.Module, module_name: str):
    parts = module_name.split('.')
    parent = model
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1]


def collect_named_linears(model: nn.Module) -> dict[str, nn.Linear]:
    return {name: module for name, module in model.named_modules() if isinstance(module, nn.Linear)}


def is_mlp_linear(name: str) -> bool:
    return '.mlp.' in name and any(name.endswith(s) for s in ['gate_proj', 'up_proj', 'down_proj'])


def is_attn_linear(name: str) -> bool:
    return '.self_attn.' in name and any(name.endswith(s) for s in ['q_proj', 'k_proj', 'v_proj', 'o_proj'])


def should_target_module(name: str) -> bool:
    if VARIANT == 'full_quant' or VARIANT == 'full_prune':
        return is_mlp_linear(name) or is_attn_linear(name)
    if 'mlp_only' in VARIANT:
        return is_mlp_linear(name)
    if 'attn_only' in VARIANT:
        return is_attn_linear(name)
    return False


def get_results_state() -> dict:
    return read_json(RESULTS_DIR / 'results_state.json', default={
        'run_name': RUN_NAME,
        'variant': VARIANT,
        'base_model': cfg['base_model'],
        'technique': TECHNIQUE_NAME,
    })


def save_results_state(state: dict):
    save_json(state, RESULTS_DIR / 'results_state.json')


def update_summary_table(state: dict) -> dict:
    summary = {
        'run_name': state.get('run_name', RUN_NAME),
        'technique': state.get('technique', TECHNIQUE_NAME),
        'variant': state.get('variant', VARIANT),
        'base_model': state.get('base_model', cfg['base_model']),
        'artifact_dir': str(ARTIFACT_DIR),
        'artifact_size_gb': artifact_size_gb(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None,
        'perplexity': state.get('perplexity', {}).get('perplexity'),
        'arc_accuracy': state.get('arc_challenge', {}).get('accuracy'),
        'artifact_zip': state.get('artifact_zip'),
        'results_zip': state.get('results_zip'),
    }
    summary_path = RESULTS_DIR / 'summary_metrics.csv'
    pd.DataFrame([summary]).to_csv(summary_path, index=False)
    return summary


def flush_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def generate_text(model, tokenizer, prompt: str, max_new_tokens: int) -> str:
    device = get_model_device(model)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=cfg['eval'].get('generation_max_input_length', 1792),
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    gen_ids = output_ids[0, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


def first_choice_letter(text: str) -> str | None:
    if not text:
        return None
    m = re.search(r'([ABCDE])', text.strip().upper())
    if m:
        return m.group(1)
    for ch in text.strip().upper():
        if ch in 'ABCDE':
            return ch
    return None


def format_arc_prompt(row):
    labels = row['choices']['label']
    texts = row['choices']['text']
    prompt = row['question'].strip() + '\n'
    for lbl, txt in zip(labels, texts):
        prompt += f'{lbl}. {txt}\n'
    prompt += 'Answer:'
    return prompt


def evaluate_perplexity_only(model, tokenizer, cfg):
    ppl_cfg = cfg['eval']['perplexity']
    logger.info(
        'Loading Perplexity dataset: %s / %s / %s | max_length=%d stride=%d max_eval_tokens=%d',
        ppl_cfg['dataset'],
        ppl_cfg['dataset_name'],
        ppl_cfg['split'],
        ppl_cfg['max_length'],
        ppl_cfg['stride'],
        ppl_cfg.get('max_eval_tokens', 131072),
    )
    ds = load_dataset(ppl_cfg['dataset'], ppl_cfg['dataset_name'], split=ppl_cfg['split'])
    text = '\n\n'.join(t for t in ds['text'] if t and t.strip())
    encodings = tokenizer(text, return_tensors='pt')
    seq_len = min(encodings.input_ids.size(1), ppl_cfg.get('max_eval_tokens', 131072))
    max_len = ppl_cfg['max_length']
    stride = ppl_cfg['stride']
    log_every = ppl_cfg.get('log_every', 25)

    nlls = []
    prev_end = 0
    start_time = time.time()
    peak_vram = 0.0
    windows = list(range(0, seq_len, stride))
    total_windows = max(1, len(windows))
    device = get_model_device(model)

    for idx, begin in enumerate(tqdm(windows, desc='Perplexity', unit='win', leave=True), start=1):
        end = min(begin + max_len, seq_len)
        input_ids = encodings.input_ids[:, begin:end].to(device)
        attention_mask = torch.ones_like(input_ids)
        target_ids = input_ids.clone()
        target_ids[:, :max(0, prev_end - begin)] = -100

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=target_ids)
            nlls.append(outputs.loss.detach().float())

        if torch.cuda.is_available():
            peak_vram = max(peak_vram, torch.cuda.max_memory_allocated() / (1024 ** 3))

        prev_end = end
        if idx % log_every == 0 or end >= seq_len:
            elapsed = time.time() - start_time
            eta = (elapsed / idx) * (total_windows - idx) if idx else 0.0
            logger.info(
                'Perplexity progress %d/%d | elapsed=%s | eta=%s',
                idx,
                total_windows,
                format_seconds(elapsed),
                format_seconds(eta),
            )
        if end >= seq_len:
            break

    perplexity = math.exp(torch.stack(nlls).mean().item())
    elapsed = time.time() - start_time
    result = {
        'dataset': f"{ppl_cfg['dataset']}/{ppl_cfg['dataset_name']}",
        'split': ppl_cfg['split'],
        'max_length': max_len,
        'stride': stride,
        'num_tokens': int(seq_len),
        'num_windows': len(nlls),
        'perplexity': perplexity,
        'elapsed_sec': elapsed,
        'elapsed_hms': format_seconds(elapsed),
        'peak_vram_gb': peak_vram if torch.cuda.is_available() else None,
    }

    save_json(result, RESULTS_DIR / 'perplexity_metrics.json')
    state = get_results_state()
    state['perplexity'] = result
    save_results_state(state)
    summary = update_summary_table(state)
    logger.info('Perplexity finished | ppl=%.4f | elapsed=%s', result['perplexity'], result['elapsed_hms'])
    display(pd.DataFrame([result]))
    display(pd.DataFrame([summary]))
    return result


def evaluate_arc_only(model, tokenizer, cfg):
    arc_cfg = cfg['eval']['arc_challenge']
    num_samples = arc_cfg['num_samples']
    max_new = arc_cfg['max_new_tokens']
    log_every = arc_cfg.get('log_every', 50)

    logger.info('Loading ARC-Challenge: %d samples, max_new_tokens=%d', num_samples, max_new)
    ds = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test').shuffle(seed=42).select(range(num_samples))

    rows = []
    correct = 0
    start_time = time.time()
    peak_vram = 0.0
    total = len(ds)

    for idx, row in enumerate(tqdm(ds, desc='ARC-Challenge', unit='q', leave=True), start=1):
        prompt = format_arc_prompt(row)
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        generated = generate_text(model, tokenizer, prompt, max_new_tokens=max_new).strip()
        pred = first_choice_letter(generated)
        gold = row['answerKey'].strip().upper()
        is_correct = pred == gold
        correct += int(is_correct)
        if torch.cuda.is_available():
            peak_vram = max(peak_vram, torch.cuda.max_memory_allocated() / (1024 ** 3))
        rows.append({
            'index': idx - 1,
            'question': row['question'],
            'prediction_text': generated,
            'predicted_label': pred,
            'gold_label': gold,
            'correct': bool(is_correct),
        })
        if idx % log_every == 0 or idx == total:
            elapsed = time.time() - start_time
            eta = (elapsed / idx) * (total - idx) if idx else 0.0
            logger.info(
                'ARC progress %d/%d | acc=%.4f | elapsed=%s | eta=%s',
                idx,
                total,
                correct / idx,
                format_seconds(elapsed),
                format_seconds(eta),
            )

    elapsed = time.time() - start_time
    result = {
        'dataset': 'arc_challenge',
        'num_samples': total,
        'max_new_tokens': max_new,
        'accuracy': correct / max(total, 1),
        'correct': correct,
        'elapsed_sec': elapsed,
        'elapsed_hms': format_seconds(elapsed),
        'peak_vram_gb': peak_vram if torch.cuda.is_available() else None,
    }
    pd.DataFrame(rows).to_csv(RESULTS_DIR / 'arc_challenge_predictions.csv', index=False)
    save_json(result, RESULTS_DIR / 'arc_challenge_metrics.json')
    state = get_results_state()
    state['arc_challenge'] = result
    save_results_state(state)
    summary = update_summary_table(state)
    results_zip = zip_directory(RESULTS_DIR, BUNDLE_DIR / f'{VARIANT}_results_only.zip')
    state['results_zip'] = str(results_zip)
    save_results_state(state)
    logger.info('ARC-Challenge finished | acc=%.4f | elapsed=%s', result['accuracy'], result['elapsed_hms'])
    display(pd.DataFrame([result]))
    display(pd.DataFrame([summary]))
    print('Results zip:', results_zip)
    return result


In [ ]:

class SmoothQuantLinear(nn.Module):
    def __init__(self, linear: nn.Linear, alpha: float = 0.8, precomputed_input_scale: torch.Tensor | None = None):
        super().__init__()
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.alpha = float(alpha)
        if linear.bias is not None:
            self.bias = nn.Parameter(linear.bias.detach().clone())
        else:
            self.bias = None
        self.register_buffer('input_scale', torch.ones(self.in_features, dtype=torch.float32))
        self.register_buffer('weight_int8', torch.zeros((self.out_features, self.in_features), dtype=torch.int8))
        self.register_buffer('weight_scale', torch.ones(self.out_features, dtype=torch.float32))
        self.register_buffer('weight_zero_point', torch.zeros(self.out_features, dtype=torch.float32))
        if precomputed_input_scale is None:
            precomputed_input_scale = torch.ones(self.in_features, dtype=torch.float32)
        self.pack(linear.weight.detach().cpu(), precomputed_input_scale)

    @torch.no_grad()
    def pack(self, weight_fp: torch.Tensor, input_scale: torch.Tensor):
        input_scale = input_scale.detach().float().clamp_min(1e-6).cpu()
        smoothed_w = weight_fp.detach().float().cpu() * input_scale.unsqueeze(0)
        row_absmax = smoothed_w.abs().amax(dim=1).clamp_min(1e-6)
        q_w = torch.clamp(torch.round(smoothed_w / row_absmax.unsqueeze(1) * 127.0), -127, 127).to(torch.int8)
        self.input_scale.copy_(input_scale)
        self.weight_int8.copy_(q_w)
        self.weight_scale.copy_(row_absmax / 127.0)
        self.weight_zero_point.zero_()

    def _dequant_weight(self, device, dtype):
        w = self.weight_int8.to(device=device, dtype=torch.float32)
        w = w * self.weight_scale.to(device=device, dtype=torch.float32).unsqueeze(1)
        return w.to(dtype=torch.float32)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        original_dtype = x.dtype
        device = x.device
        x_fp = x.float()
        scale = self.input_scale.to(device=device, dtype=torch.float32)
        x_smooth = x_fp / scale
        flat = x_smooth.reshape(-1, x_smooth.shape[-1])
        token_absmax = flat.abs().amax(dim=1, keepdim=True).clamp_min(1e-6)
        q_x = torch.clamp(torch.round(flat / token_absmax * 127.0), -127, 127)
        dq_x = q_x * (token_absmax / 127.0)
        dq_x = dq_x.reshape_as(x_smooth)
        dq_w = self._dequant_weight(device=device, dtype=original_dtype)
        y = torch.matmul(dq_x, dq_w.transpose(0, 1))
        if self.bias is not None:
            y = y + self.bias.to(device=device, dtype=torch.float32)
        return y.to(original_dtype)


def get_calibration_dataset(tokenizer, cfg):
    cal = cfg['calibration']
    logger.info(
        'Loading calibration data: %s / %s / %s | num_samples=%d max_length=%d',
        cal['dataset'], cal['dataset_name'], cal['split'], cal['num_samples'], cal['max_length']
    )
    ds = load_dataset(cal['dataset'], cal['dataset_name'], split=cal['split'])
    ds = ds.shuffle(seed=cal['seed'])
    samples = []
    for row in ds:
        text = (row.get('text') or '').strip()
        if not text:
            continue
        toks = tokenizer(text, return_tensors='pt', truncation=True, max_length=cal['max_length'])
        if toks['input_ids'].numel() < 8:
            continue
        samples.append(toks['input_ids'][0])
        if len(samples) >= cal['num_samples']:
            break
    if len(samples) < cal['num_samples']:
        logger.warning('Only collected %d calibration samples', len(samples))
    return samples


def collect_activation_maxima(model, tokenizer, cfg, target_names: list[str]) -> dict[str, torch.Tensor]:
    device = get_model_device(model)
    activation_max = {name: None for name in target_names}
    hooks = []

    def make_hook(name):
        def hook(module, args, output):
            x = args[0].detach()
            x = x.reshape(-1, x.shape[-1]).abs().amax(dim=0).cpu().float()
            prev = activation_max[name]
            activation_max[name] = x if prev is None else torch.maximum(prev, x)
        return hook

    named = dict(model.named_modules())
    for name in target_names:
        hooks.append(named[name].register_forward_hook(make_hook(name)))

    samples = get_calibration_dataset(tokenizer, cfg)
    logger.info('Collecting activation maxima for %d target linear layers', len(target_names))
    for idx, input_ids in enumerate(tqdm(samples, desc='Calibration', unit='sample', leave=True), start=1):
        input_ids = input_ids.unsqueeze(0).to(device)
        attention_mask = torch.ones_like(input_ids)
        with torch.no_grad():
            model(input_ids=input_ids, attention_mask=attention_mask)
        if idx % 32 == 0 or idx == len(samples):
            logger.info('Calibration progress %d/%d', idx, len(samples))

    for hook in hooks:
        hook.remove()

    for name in target_names:
        if activation_max[name] is None:
            mod = named[name]
            activation_max[name] = torch.ones(mod.in_features, dtype=torch.float32)
    flush_cuda()
    return activation_max


def build_input_scale(linear: nn.Linear, act_max: torch.Tensor, alpha: float) -> torch.Tensor:
    weight_max = linear.weight.detach().float().abs().amax(dim=0).clamp_min(1e-6).cpu()
    act_max = act_max.float().clamp_min(1e-6).cpu()
    scale = (act_max.pow(alpha) / weight_max.pow(1.0 - alpha)).clamp_min(1e-6)
    return scale


def apply_smoothquant_wrappers(model, activation_maxima: dict[str, torch.Tensor] | None, alpha: float):
    named_linears = collect_named_linears(model)
    target_names = [name for name in named_linears if should_target_module(name)]
    logger.info('Applying SmoothQuant wrappers to %d modules for variant=%s', len(target_names), VARIANT)
    for name in target_names:
        linear = named_linears[name]
        act_max = activation_maxima[name] if activation_maxima is not None else torch.ones(linear.in_features)
        input_scale = build_input_scale(linear, act_max, alpha)
        wrapper = SmoothQuantLinear(linear, alpha=alpha, precomputed_input_scale=input_scale)
        wrapper = wrapper.to(device=linear.weight.device, dtype=linear.weight.dtype)
        parent, attr = find_parent_module(model, name)
        setattr(parent, attr, wrapper)
    return model, target_names


def save_smoothquant_artifact(model, tokenizer, manifest: dict):
    config = AutoConfig.from_pretrained(cfg['base_model'])
    config.save_pretrained(ARTIFACT_DIR)
    tokenizer.save_pretrained(ARTIFACT_DIR)
    torch.save({'state_dict': model.state_dict(), 'manifest': manifest}, ARTIFACT_DIR / 'smoothquant_artifact.pt')
    save_json(manifest, ARTIFACT_DIR / 'smoothquant_manifest.json')
    artifact_zip = zip_directory(ARTIFACT_DIR, BUNDLE_DIR / f'{VARIANT}_artifact.zip')
    state = get_results_state()
    state['artifact_zip'] = str(artifact_zip)
    state['artifact_manifest'] = manifest
    save_results_state(state)
    logger.info('Saved SmoothQuant artifact to %s', ARTIFACT_DIR)
    return artifact_zip


def load_smoothquant_model_and_tokenizer(artifact_dir: Path):
    manifest = read_json(artifact_dir / 'smoothquant_manifest.json')
    tokenizer = AutoTokenizer.from_pretrained(artifact_dir)
    tokenizer = ensure_pad_token(tokenizer)
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        manifest['base_model'],
        torch_dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
    )
    model.eval()
    model, _ = apply_smoothquant_wrappers(model, activation_maxima=None, alpha=manifest['alpha'])
    payload = torch.load(artifact_dir / 'smoothquant_artifact.pt', map_location='cpu')
    model.load_state_dict(payload['state_dict'], strict=True)
    model.eval()
    return model, tokenizer, manifest


def quantize_variant_smoothquant(cfg):
    alpha = cfg['smoothquant']['alpha']
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'])
    tokenizer = ensure_pad_token(tokenizer)
    model = AutoModelForCausalLM.from_pretrained(
        cfg['base_model'],
        torch_dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
    )
    model.eval()

    named_linears = collect_named_linears(model)
    target_names = [name for name in named_linears if should_target_module(name)]
    if not target_names:
        raise RuntimeError(f'No target modules matched for variant={VARIANT}')
    logger.info('Target modules selected: %d', len(target_names))

    activation_maxima = collect_activation_maxima(model, tokenizer, cfg, target_names)
    model, target_names = apply_smoothquant_wrappers(model, activation_maxima=activation_maxima, alpha=alpha)

    manifest = {
        'technique': TECHNIQUE_NAME,
        'variant': VARIANT,
        'base_model': cfg['base_model'],
        'alpha': alpha,
        'target_modules': target_names,
        'calibration': cfg['calibration'],
        'notes': 'Dynamic per-token activation quantization with per-row static weight quantization. Implemented with a notebook-local SmoothQuantLinear wrapper for Colab portability.',
    }
    artifact_zip = save_smoothquant_artifact(model, tokenizer, manifest)
    summary = update_summary_table(get_results_state())
    display(pd.DataFrame([summary]))
    print('Artifact zip:', artifact_zip)
    return manifest


## Cell 1 — Build and save the SmoothQuant artifact

In [ ]:
quant_manifest = quantize_variant_smoothquant(cfg)
print(json.dumps(quant_manifest, indent=2))

## Cell 2 — Reload the artifact and evaluate perplexity

In [ ]:
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(f'Artifact not found: {ARTIFACT_DIR}')
model, tokenizer, manifest = load_smoothquant_model_and_tokenizer(ARTIFACT_DIR)
try:
    perplexity_result = evaluate_perplexity_only(model, tokenizer, cfg)
finally:
    flush_cuda()

The tokenizer you are loading from '/content/artifacts/smoothquant/full_quant' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


2026-03-10 05:55:56,810 | INFO    | We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
2026-03-10 05:55:58,373 | INFO    | Applying SmoothQuant wrappers to 196 modules for variant=full_quant
2026-03-10 05:56:05,644 | INFO    | Loading Perplexity dataset: wikitext / wikitext-2-raw-v1 / test | max_length=2048 stride=1024 max_eval_tokens=131072


Token indices sequence length is longer than the specified maximum sequence length for this model (298938 > 32768). Running this sequence through the model will result in indexing errors


Perplexity:   0%|          | 0/128 [00:00<?, ?win/s]

2026-03-10 05:56:33,025 | INFO    | Perplexity progress 25/128 | elapsed=00:00:22 | eta=00:01:34
2026-03-10 05:56:56,589 | INFO    | Perplexity progress 50/128 | elapsed=00:00:46 | eta=00:01:12
2026-03-10 05:57:20,817 | INFO    | Perplexity progress 75/128 | elapsed=00:01:10 | eta=00:00:49
2026-03-10 05:57:45,009 | INFO    | Perplexity progress 100/128 | elapsed=00:01:34 | eta=00:00:26
2026-03-10 05:58:08,905 | INFO    | Perplexity progress 125/128 | elapsed=00:01:58 | eta=00:00:02
2026-03-10 05:58:10,808 | INFO    | Perplexity progress 127/128 | elapsed=00:02:00 | eta=00:00:00
2026-03-10 05:58:11,070 | INFO    | Perplexity finished | ppl=8.4559 | elapsed=00:02:00


,dataset,split,max_length,stride,num_tokens,num_windows,perplexity,elapsed_sec,elapsed_hms,peak_vram_gb
0,wikitext/wikitext-2-raw-v1,test,2048,1024,131072,127,8.45594,120.891819,00:02:00,5.26473


,run_name,technique,variant,base_model,artifact_dir,artifact_size_gb,perplexity,arc_accuracy,artifact_zip,results_zip
0,smoothquant_full_quant_ppl_arc,SmoothQuant,full_quant,Qwen/Qwen2-1.5B,/content/artifacts/smoothquant/full_quant,1.67364,8.45594,None,/content/zip_bundles/smoothquant_full_quant_pp...,None


## Cell 3 — Reload the artifact and evaluate ARC-Challenge

In [ ]:
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(f'Artifact not found: {ARTIFACT_DIR}')
model, tokenizer, manifest = load_smoothquant_model_and_tokenizer(ARTIFACT_DIR)
try:
    arc_result = evaluate_arc_only(model, tokenizer, cfg)
finally:
    flush_cuda()

The tokenizer you are loading from '/content/artifacts/smoothquant/full_quant' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


2026-03-10 06:11:15,042 | INFO    | We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
2026-03-10 06:11:16,487 | INFO    | Applying SmoothQuant wrappers to 196 modules for variant=full_quant
2026-03-10 06:11:25,133 | INFO    | Loading ARC-Challenge: 500 samples, max_new_tokens=5


README.md: 0.00B [00:00, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…):   0%|          | 0.00/190k [00:00<?, ?B/s]

ARC-Challenge/test-00000-of-00001.parque(…):   0%|          | 0.00/204k [00:00<?, ?B/s]

...

ARC-Challenge:   0%|          | 0/500 [00:00<?, ?q/s]

2026-03-10 06:11:57,964 | INFO    | ARC progress 50/500 | acc=0.6600 | elapsed=00:00:25 | eta=00:03:53
2026-03-10 06:12:23,669 | INFO    | ARC progress 100/500 | acc=0.6600 | elapsed=00:00:51 | eta=00:03:26
2026-03-10 06:12:49,271 | INFO    | ARC progress 150/500 | acc=0.6467 | elapsed=00:01:17 | eta=00:03:00
2026-03-10 06:13:14,917 | INFO    | ARC progress 200/500 | acc=0.6350 | elapsed=00:01:42 | eta=00:02:34
2026-03-10 06:13:40,711 | INFO    | ARC progress 250/500 | acc=0.6520 | elapsed=00:02:08 | eta=00:02:08
2026-03-10 06:14:06,482 | INFO    | ARC progress 300/500 | acc=0.6500 | elapsed=00:02:34 | eta=00:01:42
2026-03-10 06:14:32,132 | INFO    | ARC progress 350/500 | acc=0.6457 | elapsed=00:03:00 | eta=00:01:17
2026-03-10 06:14:57,858 | INFO    | ARC progress 400/500 | acc=0.6500 | elapsed=00:03:25 | eta=00:00:51
2026-03-10 06:15:23,629 | INFO    | ARC progress 450/500 | acc=0.6511 | elapsed=00:03:51 | eta=00:00:25
2026-03-10 06:15:49,406 | INFO    | ARC progress 500/500 | acc=0.

,dataset,num_samples,max_new_tokens,accuracy,correct,elapsed_sec,elapsed_hms,peak_vram_gb
0,arc_challenge,500,5,0.646,323,257.389955,00:04:17,1.807884


,run_name,technique,variant,base_model,artifact_dir,artifact_size_gb,perplexity,arc_accuracy,artifact_zip,results_zip
0,smoothquant_full_quant_ppl_arc,SmoothQuant,full_quant,Qwen/Qwen2-1.5B,/content/artifacts/smoothquant/full_quant,1.67364,8.45594,0.646,/content/zip_bundles/smoothquant_full_quant_pp...,None


Results zip: /content/zip_bundles/smoothquant_full_quant_ppl_arc/full_quant/full_quant_results_only.zip


## Final summary and saved files

In [ ]:
state = get_results_state()
summary = update_summary_table(state)
print(json.dumps(state, indent=2))
display(pd.DataFrame([summary]))
print('\nSaved files under results:')
for p in sorted(RESULTS_DIR.glob('*')):
    print('-', p.name)
print('\nSaved files under artifacts:')
for p in sorted(ARTIFACT_DIR.glob('*')):
    print('-', p.name)
print('\nSaved zip bundles:')
for p in sorted(BUNDLE_DIR.glob('*')):
    print('-', p.name)

{
  "run_name": "smoothquant_full_quant_ppl_arc",
  "variant": "full_quant",
  "base_model": "Qwen/Qwen2-1.5B",
  "technique": "SmoothQuant",
  "artifact_zip": "/content/zip_bundles/smoothquant_full_quant_ppl_arc/full_quant/full_quant_artifact.zip",
  "artifact_manifest": {
    "technique": "SmoothQuant",
    "variant": "full_quant",
    "base_model": "Qwen/Qwen2-1.5B",
    "alpha": 0.8,
    "target_modules": [
      "model.layers.0.self_attn.q_proj",
      "model.layers.0.self_attn.k_proj",
      "model.layers.0.self_attn.v_proj",
      "model.layers.0.self_attn.o_proj",
      "model.layers.0.mlp.gate_proj",
      "model.layers.0.mlp.up_proj",
      "model.layers.0.mlp.down_proj",
      "model.layers.1.self_attn.q_proj",
      "model.layers.1.self_attn.k_proj",
      "model.layers.1.self_attn.v_proj",
      "model.layers.1.self_attn.o_proj",
      "model.layers.1.mlp.gate_proj",
      "model.layers.1.mlp.up_proj",
      "model.layers.1.mlp.down_proj",
      "model.layers.2.self_attn.q

,run_name,technique,variant,base_model,artifact_dir,artifact_size_gb,perplexity,arc_accuracy,artifact_zip,results_zip
0,smoothquant_full_quant_ppl_arc,SmoothQuant,full_quant,Qwen/Qwen2-1.5B,/content/artifacts/smoothquant/full_quant,1.67364,8.45594,0.646,/content/zip_bundles/smoothquant_full_quant_pp...,/content/zip_bundles/smoothquant_full_quant_pp...



Saved files under results:
- arc_challenge_metrics.json
- arc_challenge_predictions.csv
- perplexity_metrics.json
- results_state.json
- summary_metrics.csv

Saved files under artifacts:
- added_tokens.json
- chat_template.jinja
- config.json
- merges.txt
- smoothquant_artifact.pt
- smoothquant_manifest.json
- special_tokens_map.json
- tokenizer.json
- tokenizer_config.json
- vocab.json

Saved zip bundles:
- full_quant_artifact.zip
- full_quant_results_only.zip
